# Airbnb Price Analysis in New York

En este proyecto analizaremos el mercado de Airbnb en Nueva York para entender qué factores influyen en el precio de los alojamientos.

In [78]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


## 1. Importación de librerías

In [79]:
import pandas as pd

## 2. Carga del dataset

In [80]:
df = pd.read_csv("../data/AB_NYC_2019.csv")
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


## 3. Exploración inicial

El objetivo de esta sección es comprender la estructura del dataset e identificar posibles problemas de calidad antes de comenzar el análisis.

In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  str    
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  str    
 4   neighbourhood_group             48895 non-null  str    
 5   neighbourhood                   48895 non-null  str    
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  str    
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     38843 non-n

In [82]:
df.isnull().sum()

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [83]:
df.duplicated().sum()

np.int64(0)

In [84]:
df.describe()

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,4.889500e+04,4.889500e+04,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,38843.000000,48895.000000,48895.000000
mean,1.901714e+07,6.762001e+07,40.728949,-73.952170,152.720687,7.029962,23.274466,1.373221,7.143982,112.781327
std,1.098311e+07,7.861097e+07,0.054530,0.046157,240.154170,20.510550,44.550582,1.680442,32.952519,131.622289
min,2.539000e+03,2.438000e+03,40.499790,-74.244420,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.471945e+06,7.822033e+06,40.690100,-73.983070,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.967728e+07,3.079382e+07,40.723070,-73.955680,106.000000,3.000000,5.000000,0.720000,1.000000,45.000000
75%,2.915218e+07,1.074344e+08,40.763115,-73.936275,175.000000,5.000000,24.000000,2.020000,2.000000,227.000000
max,3.648724e+07,2.743213e+08,40.913060,-73.712990,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


### Conclusiones:
- Las columnas last review y reviews per month tienen demasiados valores nulos, buscar posible autocorrelación entre ellas.

- Host name es irrelevante, host id es la columna importante, además se puede añadir la cantidad de propiedades que tiene cada host como nueva columna.

- no tiene sentido que price pueda ser igual a cero

- no tiene sentido que noches minimas sean 1250

## Limpieza de datos

### Tratar nulos

In [85]:
# Habia decidido eliminar la columna host_name, por lo que los valores nulos de esta columna van a ser eliminados junto a ella.
# Aprovecho para eliminar más columnas que considerare poco importantes para mi analisis.
# Eliminaré también name ya que considero que tambien será irrelevante
# id también la voy a eliminar

df_clean = df.drop(columns=['name', 'host_name', 'id'])

In [86]:
# ¿Los valores nulos de last_review y reviews_per_month corresponden a alojamientos que nunca han recibido una reseña.?

df[(df["number_of_reviews"] == 0) & (df["last_review"].notna())]


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365


In [87]:
df[(df["number_of_reviews"] == 0) & (df["reviews_per_month"].notna())]

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365


In [88]:
df[(df["number_of_reviews"] > 0) & (df["last_review"].isna())]

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365


In [89]:
df[(df["number_of_reviews"] > 0) & (df["reviews_per_month"].isna())]

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365


Se comprobó que los valores nulos de last_review y reviews_per_month corresponden exclusivamente a alojamientos sin reseñas (number_of_reviews = 0). Por tanto, estos nulos representan la ausencia de reseñas y no un error o pérdida de información.

In [90]:
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0)
# Aún no se si utilizare la columna de ultima reseña asi que por ahora no la voy a eliminar

### Datos atípicos

In [91]:

(df['price'] == 0).sum()

np.int64(11)

In [92]:
# al ser solo 11 voy rellenarlos con la mediana
df_clean.loc[df_clean['price'] == 0, 'price'] = df_clean['price'].median()



In [93]:
(df['minimum_nights'] == 1250).sum()

np.int64(1)

In [94]:
df_clean["minimum_nights"].describe()

count    48895.000000
mean         7.029962
std         20.510550
min          1.000000
25%          1.000000
50%          3.000000
75%          5.000000
max       1250.000000
Name: minimum_nights, dtype: float64

In [95]:
df_clean["minimum_nights"].value_counts().sort_index().head(20)

minimum_nights
1     12720
2     11696
3      7999
4      3303
5      3034
6       752
7      2058
8       130
9        80
10      483
11       33
12       91
13       54
14      562
15      279
16       18
17       14
18       28
19        8
20      223
Name: count, dtype: int64

In [96]:
df_clean["minimum_nights"].nlargest(20)

5767     1250
2854     1000
13404     999
26341     999
38664     999
7355      500
8014      500
11193     500
14285     500
47620     500
10829     480
34487     400
1305      370
15946     366
700       365
754       365
1449      365
2150      365
2831      365
3398      365
Name: minimum_nights, dtype: int64

he analizado los valores de minimum_nights y, aunque existen estancias mínimas elevadas (hasta 47 noches), estas pueden corresponder a alquileres de larga duración y aparecen en un número significativo de alojamientos. Al no existir evidencia de que sean errores, se decide mantener la variable sin modificaciones

In [97]:
df_clean.isnull().sum()

host_id                               0
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                     0
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [98]:
df_clean.describe()
#Aunque el precio mas alto me parezca excesivamente alto, voy a dejarlo asi ya que puede ser algun tipo de casa de lujo...

,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,4.889500e+04,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000
mean,6.762001e+07,40.728949,-73.952170,152.744534,7.029962,23.274466,1.090910,7.143982,112.781327
std,7.861097e+07,0.054530,0.046157,240.144266,20.510550,44.550582,1.597283,32.952519,131.622289
min,2.438000e+03,40.499790,-74.244420,10.000000,1.000000,0.000000,0.000000,1.000000,0.000000
25%,7.822033e+06,40.690100,-73.983070,69.000000,1.000000,1.000000,0.040000,1.000000,0.000000
50%,3.079382e+07,40.723070,-73.955680,106.000000,3.000000,5.000000,0.370000,1.000000,45.000000
75%,1.074344e+08,40.763115,-73.936275,175.000000,5.000000,24.000000,1.580000,2.000000,227.000000
max,2.743213e+08,40.913060,-73.712990,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


## Conclusiones de la limpieza

- Los valores nulos de last_review corresponden a alojamientos sin reseñas y se mantienen.
- Los valores nulos de reviews_per_month se sustituyen por 0 al representar ausencia de reseñas.
- No se detectan duplicados.
- No se modifican los valores elevados de minimum_nights, ya que representan estancias largas.
- Los valores extremos de price se conservarán para analizarlos en la siguiente fase.